In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")

freq = pd.read_csv(DATA_DIR / "freMTPLfreq.csv")
sev = pd.read_csv(DATA_DIR / "freMTPLsev.csv")

sev_agg = (
    sev.groupby("PolicyID", as_index=False)
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),
           ClaimCountFromSev=("ClaimAmount", "size")
       )
)

df = freq.merge(sev_agg, on="PolicyID", how="left")
df["ClaimAmount"] = df["ClaimAmount"].fillna(0.0)
df["ClaimCountFromSev"] = df["ClaimCountFromSev"].fillna(0).astype(int)

In [14]:
# mild outlier / data-quality treatment
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
df["Exposure"] = df["Exposure"].clip(lower=0, upper=1)
df["ClaimAmount"] = df["ClaimAmount"].clip(upper=200_000)

# if total claim amount is zero, force count to zero
df.loc[(df["ClaimAmount"] == 0) & (df["ClaimNb"] > 0), "ClaimNb"] = 0

# drop impossible exposure rows
df = df[df["Exposure"] > 0].copy()

# small feature engineering
df["CarAgeCapped"] = df["CarAge"].clip(upper=20)
df["DriverAgeCapped"] = df["DriverAge"].clip(lower=18, upper=90)
df["LogDensity"] = np.log1p(df["Density"])

# modeling targets
df["Frequency"] = df["ClaimNb"] / df["Exposure"]
df["AvgClaimAmount"] = df["ClaimAmount"] / np.maximum(df["ClaimNb"], 1)
df["PurePremium"] = df["ClaimAmount"] / df["Exposure"]
df["HasClaim"] = (df["ClaimAmount"] > 0).astype(int)

In [15]:
print(df.shape)
print(df[["ClaimNb", "ClaimCountFromSev", "ClaimAmount", "Exposure"]].describe())
print("Policies with claims:", df["HasClaim"].sum())
print("Mismatch rate ClaimNb vs matched claim rows:",
      (df["ClaimNb"] != df["ClaimCountFromSev"]).mean())

(413169, 19)
             ClaimNb  ClaimCountFromSev    ClaimAmount       Exposure
count  413169.000000      413169.000000  413169.000000  413169.000000
mean        0.039163           0.039163      75.051942       0.560982
std         0.204053           0.204053    1575.095435       0.369304
min         0.000000           0.000000       0.000000       0.002732
25%         0.000000           0.000000       0.000000       0.200000
50%         0.000000           0.000000       0.000000       0.540000
75%         0.000000           0.000000       0.000000       1.000000
max         4.000000           4.000000  200000.000000       1.000000
Policies with claims: 15390
Mismatch rate ClaimNb vs matched claim rows: 0.0
